In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
df =spark.read.table("shopkart.bronze.orders_dirty")

In [0]:
df.dropDuplicates(["order_id"])

In [0]:
df.select("order_id").distinct().count()

In [0]:
df = df.fillna('Unknown', subset=['order_status'])

In [0]:
display(df)

In [0]:
df.filter(df.order_status == "Unknown").count()

In [0]:
df1 = df.withColumn('order_status',trim(lower('order_status')))

In [0]:
display(df1)

In [0]:
df1.filter(col("order_status") != lower(trim("order_status"))).count()

In [0]:
df.select("order_id").distinct().count()

In [0]:
df1.printSchema()


In [0]:
df1.withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast("timestamp"))


In [0]:
#df1.withColumn("order_approved_at", col("order_purchase_timestamp").cast("timestamp"))
df1.withColumn("order_delivered_carrier_date", col("order_purchase_timestamp").cast("timestamp"))
df1.withColumn("order_delivered_customer_date", col("order_purchase_timestamp").cast("timestamp"))
df1.withColumn("order_estimated_delivery_date", col("order_purchase_timestamp").cast("timestamp"))
df1.withColumn("_extracted_at", col("order_purchase_timestamp").cast("timestamp"))

In [0]:
df2 = df1.withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast("timestamp"))

In [0]:
df2.printSchema()

In [0]:
df3 = df2.withColumn("order_approved_at", col("order_approved_at").cast("timestamp"))

In [0]:
df3.printSchema()

In [0]:
df4=df3.withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast("timestamp"))

In [0]:
df4.printSchema()

In [0]:
df5=df4.withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast("timestamp"))

In [0]:
df6=df5.withColumn("_extracted_at", col("_extracted_at").cast("timestamp"))

In [0]:
df6.printSchema()

In [0]:
df6.withColumn("delivery_days",lit("none")).show()

In [0]:
df7 =df6.withColumn("delivery_days", when(col("order_delivered_customer_date") < col("order_purchase_timestamp"), "is_valid_delivery")
.otherwise("None")
)

In [0]:
df7.filter(df7.delivery_days == "is_valid_delivery").show()

In [0]:
df7.write.format("delta").mode("overwrite").option("header","true").saveAsTable("shopkart.silver.orders")

In [0]:
# ---- SK-04: "before" profile of bronze.orders_dirty (run this again on silver.orders for "after") ----
from pyspark.sql import functions as F

df = spark.table("shopkart.silver.orders")
total = df.count()
profile = {
    "total rows":                total,
    "distinct order_id":         df.select("order_id").distinct().count(),
    "exact duplicate rows":      total - df.distinct().count(),
    "NULL order_status":         df.filter(F.col("order_status").isNull()).count(),
    "non-normalised status":     df.filter(F.col("order_status") != F.lower(F.trim("order_status"))).count(),
    "delivered before purchase": df.filter(F.to_timestamp("order_delivered_customer_date")
                                           < F.to_timestamp("order_purchase_timestamp")).count(),
}
profile["key duplicate rows (total - distinct ids)"] = total - profile["distinct order_id"]
for k, v in profile.items():
    print(f"{k:<45} {v:>7,}")